# 00 · Load check

Proves the environment works and the data is what we think it is. No modeling.

**Dataset:** Sparkov simulated card transactions (`kartik2112/fraud-detection`) — 1,000 customers,
800 merchants, two years. It replaced the ULB `creditcard.csv` dataset at the start of Phase 1;
see `plans/phase-1-modeling-plan.md` for why.

In [1]:
from fraud_radar import data

df = data.load_all()
df.shape

(1852394, 18)

### Shape and time span

Expected: **1,852,394** transactions across 21 columns, spanning 2019-01-01 to 2020-12-31.

In [2]:
assert len(df) == 1_852_394, f"unexpected row count: {len(df)}"
print(f"rows      {len(df):,}")
print(f"columns   {list(df.columns)}")
print(f"span      {df.ts.min().date()}  ->  {df.ts.max().date()}")
print(f"cards     {df.cc_num.nunique():,}")
print(f"merchants {df.merchant.nunique():,}")

rows      1,852,394
columns   ['ts', 'cc_num', 'merchant', 'category', 'amt', 'gender', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'merch_lat', 'merch_long', 'is_fraud', 'split']
span      2019-01-01  ->  2020-12-31
cards     999
merchants 693


### Class balance

Fraud is rare, but nothing like as rare as in the ULB dataset (0.17%). Around **0.5%** here, which
is still imbalanced enough that accuracy is useless.

In [3]:
for split in ["train", "test"]:
    s = df[df.split == split]
    print(f"{split:5s}  {len(s):>9,} rows   {s.is_fraud.sum():>5,} fraud   {s.is_fraud.mean():.4%}"
          f"   {s.ts.min().date()} -> {s.ts.max().date()}")

train  1,296,675 rows   7,506 fraud   0.5789%   2019-01-01 -> 2020-06-21


test     555,719 rows   2,145 fraud   0.3860%   2020-06-21 -> 2020-12-31


**The split is chronological, and that matters.** The dataset ships train and test already
divided in time rather than at random: train ends 2020-06-21, test begins there. A model is
therefore always predicting forward, which is the only thing a deployed system can do.

Note the fraud rate **falls from 0.579% to 0.386%** between the two periods. That is real drift —
the thing being predicted changes over time — and it is what makes the Phase 5 drift stretch goal
possible on this dataset.

In [4]:
naive = (df.is_fraud == 0).mean()
print(f'a model answering "never fraud" scores {naive:.4%} accuracy and catches 0 of {df.is_fraud.sum():,} frauds')

a model answering "never fraud" scores 99.4790% accuracy and catches 0 of 9,651 frauds


### Missing values

In [5]:
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "no missing values in the raw data")

no missing values in the raw data


---
Environment works, data is intact. On to the EDA in `01-eda.ipynb`.